# Clase 187 — Diseño experimental

Más allá del A/B simple: **factorial** (varios factores + interacciones), **bloqueo** (reducir varianza), **fraccional** (menos corridas), **cluster randomization** (unidad de tratamiento ≠ unidad de análisis) y **switchback**. Todo apoyado en la asunción SUTVA.

Requiere: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`.

## 1. Factorial 2²

Testeamos dos factores a la vez y recuperamos efectos principales + interacción con un OLS `y ~ A*B`.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

nper = 1000
rows = []
for A in (0, 1):
    for Bf in (0, 1):
        ctr = 0.10 + 0.02*A + 0.015*Bf + 0.02*A*Bf + rng.normal(0, 0.05, nper)
        for v in ctr:
            rows.append((A, Bf, v))
df = pd.DataFrame(rows, columns=["A", "B", "ctr"])
m = smf.ols("ctr ~ A * B", data=df).fit()
print(m.params.round(4))
print("\nverdaderos: Intercept=0.10, A=0.02, B=0.015, A:B=0.02")
assert abs(m.params["A:B"] - 0.02) < 0.02

## 2. Bloqueo: reducir el SE del efecto

Bloqueando por país (variable nuisance de alta varianza), el SE del efecto de tratamiento cae.

In [ ]:
paises = {"AR": 0.5, "BR": 0.3, "MX": 0.4}
delta = 0.05
rows = []
for pais, p0 in paises.items():
    for _ in range(500):
        t = rng.integers(0, 2)
        rows.append((pais, t, rng.normal(p0 + delta*t, 0.1)))
dd = pd.DataFrame(rows, columns=["pais", "t", "y"])
se_naive = smf.ols("y ~ t", data=dd).fit().bse["t"]
se_block = smf.ols("y ~ t + C(pais)", data=dd).fit().bse["t"]
print(f"SE(δ) sin bloquear = {se_naive:.4f}")
print(f"SE(δ) bloqueado    = {se_block:.4f}")
assert se_block < se_naive

## 3. Fraccional 2^(4-1)

Con el generador `D = ABC` corremos 8 combinaciones en vez de 16. El precio: `D` queda confundido (aliased) con la interacción triple `ABC`.

In [ ]:
full = np.array(list(itertools.product([-1, 1], repeat=3)))   # A, B, C
D = full[:, 0] * full[:, 1] * full[:, 2]                       # D = ABC
design = np.column_stack([full, D])
print("Diseño 2^(4-1) — 8 corridas (vs 16 del full factorial):")
print(pd.DataFrame(design, columns=list("ABCD")))
print("\nGenerador I = ABCD  =>  D aliased con ABC, A con BCD, AB con CD ...")
assert design.shape == (8, 4)

## 4. Cluster randomization

Cuando se aleatorizan grupos (aulas, ciudades), analizar a nivel individuo infla la significancia por la correlación intra-cluster. El análisis correcto es a nivel cluster.

In [ ]:
n_clusters, m_per, icc, effect = 50, 30, 0.10, 0.3
sd_between, sd_within = np.sqrt(icc), np.sqrt(1 - icc)
rows = []
for c in range(n_clusters):
    t = c % 2
    u = rng.normal(0, sd_between)        # efecto aleatorio del cluster
    for _ in range(m_per):
        rows.append((c, t, effect*t + u + rng.normal(0, sd_within)))
dc = pd.DataFrame(rows, columns=["cluster", "t", "y"])
p_naive = smf.ols("y ~ t", data=dc).fit().pvalues["t"]
agg = dc.groupby(["cluster", "t"])["y"].mean().reset_index()
p_cluster = stats.ttest_ind(agg[agg.t == 1].y, agg[agg.t == 0].y).pvalue
print(f"p ingenuo (n=1500) = {p_naive:.4f}   p a nivel cluster (n=50) = {p_cluster:.4f}")
print("El ingenuo sobre-estima la significancia por correlación intra-cluster.")
assert p_cluster > p_naive

## 5. Switchback

Asignamos el tratamiento global por hora durante 7 días con estacionalidad horaria fuerte, y estimamos el efecto con un OLS que **controla la hora del día** (efectos fijos horarios), aislando el efecto de la estacionalidad.

In [ ]:
hours = np.arange(7 * 24)
hour_of_day = hours % 24
season = 3 * np.sin(2 * np.pi * hour_of_day / 24)
treat = rng.integers(0, 2, len(hours))     # asignación por bloque horario
y = 10 + season + 0.8 * treat + rng.normal(0, 0.5, len(hours))
sb = pd.DataFrame({"h": hour_of_day, "t": treat, "y": y})

model = smf.ols("y ~ t + C(h)", data=sb).fit()   # controla la estacionalidad horaria
print(f"efecto switchback (OLS con hora-del-día): coef={model.params['t']:.3f}  p={model.pvalues['t']:.2e}")
print("verdadero = 0.80")
assert model.pvalues["t"] < 0.05

sb["y_adj"] = sb["y"] - season      # y ajustado por estacionalidad
plt.figure(figsize=(6, 4))
for tv, lbl in ((0, "A"), (1, "B")):
    plt.hist(sb[sb.t == tv].y_adj, bins=15, alpha=0.5, label=lbl)
plt.legend(); plt.xlabel("y ajustado por estacionalidad")
plt.title("Switchback: B desplazado ~+0.8 respecto a A")
plt.tight_layout(); plt.show()

## Ejercicios

1. Extendé el factorial a 2³ (`ctr ~ A*B*C`) y verificá con `anova_lm` qué términos son significativos.
2. Calculá el `n` efectivo por cluster: `n_eff = n / (1 + (m-1)·ρ)` con `ρ=0.10, m=30` y contrastalo con el `n` nominal.
3. Discutí qué diseño usarías ante interferencia de red (efectos que se propagan entre usuarios).

## Conclusiones

- El factorial testea varios factores con menos corridas que A/B por factor **y** detecta interacciones.
- El bloqueo reduce la varianza residual si la variable de bloqueo importa.
- El fraccional cambia corridas por aliasing de efectos de alto orden (asumidos ≈ 0).
- Con contaminación entre unidades (SUTVA violada), usá cluster randomization o switchback; analizá a la unidad correcta.